# Part 1 — Ingest, Validate, Clean, Enrich
CareLoop Clinics — No-Show Guard. Input: `data/appointments.csv` (1,020 raw bookings, Jan–Jun 2025).

## Task 1.1 — Read the file with the csv module
Read the raw file without pandas. Convert the number columns and set aside the rows that cannot be converted.

In [ ]:
import csv
import json

# Open the raw file with the csv module (no pandas, as the brief asks)
with open("data/appointments.csv", newline="") as f:
    reader = csv.DictReader(f)      # each row becomes a dictionary {column: value}
    rows = list(reader)             # read every row into a list
    header = reader.fieldnames      # column names from the first line

print("Header:", header)
print("Row count:", len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020


In [ ]:
def parse_row(row):
    """Convert number columns from text to int/float.
    A blank or non-number value makes int()/float() raise ValueError."""
    new_row = dict(row)                                   # copy, so the original row stays untouched
    new_row["age"] = int(row["age"])
    new_row["fee_inr"] = int(row["fee_inr"])
    new_row["prior_no_shows"] = int(row["prior_no_shows"])
    new_row["distance_km"] = float(row["distance_km"])
    return new_row

accepted = []
rejected = []

for row in rows:
    try:
        accepted.append(parse_row(row))   # conversion worked -> keep it
    except ValueError:
        rejected.append(row)              # conversion failed -> keep the raw row aside

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

# Why were rows rejected?
print("Rejected - blank age:", sum(1 for r in rejected if r["age"] == ""))
print("Rejected - blank distance_km:", sum(1 for r in rejected if r["distance_km"] == ""))

# A type check is not a validity check: impossible ages still pass int()
print("Impossible ages accepted:", [r["age"] for r in accepted if r["age"] < 1 or r["age"] > 100])

Accepted: 988
Rejected: 32
Rejected - blank age: 12
Rejected - blank distance_km: 20
Impossible ages accepted: [0, 150, -4, 250, 199]


In [ ]:
# Save the rejected rows so the problem rows can be reviewed later
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)      # indent=2 makes the file easy to read

print("Saved rejected_rows.json with", len(rejected), "rows")

Saved rejected_rows.json with 32 rows


**Result:** 1,020 rows read → 988 accepted, 32 rejected (12 blank age, 20 blank distance).
5 impossible ages (0, 150, −4, 250, 199) passed the type check. They are fixed in Task 1.3.

## Task 1.2 — Inspect and clean, pass 1
Start again from the original file with pandas. Fix duplicates, channel spellings and dates, then drop rows that can't be used.

In [ ]:
import pandas as pd
import numpy as np

# Start again from the ORIGINAL file (Task 1.1 was a separate csv-module exercise)
df = pd.read_csv("data/appointments.csv")

print("Shape:", df.shape)
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())

Shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values per column:
appointment_id      0
patient_id          0
cl

In [ ]:
# (a) Drop exact duplicate rows (every column identical)
df = df.drop_duplicates()
print("After (a) drop duplicates:", len(df))

# (b) Clean booking_channel: remove extra spaces, then same capitalisation
print("\nbooking_channel spellings before:", df["booking_channel"].nunique())
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("booking_channel spellings after:", df["booking_channel"].nunique(), df["booking_channel"].unique())
print("After (b) clean channel:", len(df))

# (c) Convert dates. errors="coerce" turns unreadable dates into NaT (missing)
raw_appt_date = df["appt_date"].copy()                 # keep the raw text to see what failed
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("\nMissing appt_date after conversion:", df["appt_date"].isna().sum())
print("Raw values that failed:", raw_appt_date[df["appt_date"].isna()].tolist())
print("After (c) convert dates:", len(df))

# (d) Drop rows with no usable appointment date or no status
print("\nMissing status:", df["status"].isna().sum())
df = df.dropna(subset=["appt_date", "status"]).copy()  # .copy() avoids SettingWithCopyWarning later
print("After (d) drop missing date/status:", len(df))

After (a) drop duplicates: 1000

booking_channel spellings before: 16
booking_channel spellings after: 3 ['Phone' 'Walk-In' 'App']
After (b) clean channel: 1000

Missing appt_date after conversion: 6
Raw values that failed: ['not booked', '2025-06-31', '2025-04-31', 'TBD', nan, '2025-02-30']
After (c) convert dates: 1000

Missing status: 4
After (d) drop missing date/status: 990


**Result:** 1,020 → 1,000 (20 duplicates) → 990 (6 unreadable dates + 4 blank statuses).
booking_channel: 16 raw spellings → 3 (App, Phone, Walk-In).
Unreadable dates included impossible days (2025-02-30, 2025-04-31, 2025-06-31) and text (TBD, not booked, N/A).

## Task 1.3 — Clean, pass 2: validity, gaps and outliers
Order matters: fix impossible values → fill gaps → then compute quartiles and cap outliers.

In [ ]:
# (a) Validity check: an age below 1 or above 100 is impossible -> set it to missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", invalid_age.sum(), df.loc[invalid_age, "age"].tolist())
df.loc[invalid_age, "age"] = np.nan
print("Missing ages now (12 blank + 5 invalid):", df["age"].isna().sum())

# (b) Fill gaps with the median (the median is not pulled by extreme values)
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print("\nMedian age:", age_median)
print("Median distance_km:", dist_median)

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)
print("Missing after filling -> age:", df["age"].isna().sum(), "| distance_km:", df["distance_km"].isna().sum())

Invalid ages: 5 [0.0, 150.0, -4.0, 250.0, 199.0]
Missing ages now (12 blank + 5 invalid): 17

Median age: 48.0
Median distance_km: 8.25
Missing after filling -> age: 0 | distance_km: 0


In [ ]:
# (c) Cap outliers at the IQR fences (done AFTER filling gaps, as the brief asks)
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Count BEFORE capping, otherwise the count is always 0
    n_low = (df[col] < lower).sum()
    n_high = (df[col] > upper).sum()

    df[col] = np.clip(df[col], lower, upper)   # below lower -> lower, above upper -> upper

    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower={lower}, upper={upper}")
    print(f"   capped: {n_low} low, {n_high} high\n")

distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower=-7.9375, upper=25.5625
   capped: 0 low, 26 high

fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower=68.75, upper=1218.75
   capped: 0 low, 3 high



In [ ]:
# (d) Round to sensible units
df["distance_km"] = df["distance_km"].round(1)          # 1 decimal place
df["fee_inr"] = df["fee_inr"].round().astype(int)       # whole rupees

print("Largest distance_km:", df["distance_km"].max())
print("Largest fee_inr:", df["fee_inr"].max())

Largest distance_km: 25.6
Largest fee_inr: 1219


**Result:** 5 impossible ages set to missing → 17 missing ages, filled with the median 48. 20 missing distances filled with the median 8.25.
- distance_km: Q1 4.625, Q3 13.0, upper fence 25.5625 → 26 values capped (none on the low side).
- fee_inr: Q1 500, Q3 787.5, upper fence 1,218.75 → 3 values capped (the ×10 typing errors 4,500 / 5,500 / 7,500). Largest fee is now 1,219.

Note: the distance cap also pulls in some realistic far distances (26–38 km), not only the obvious typos (86, 180, 221 km). The brief asks for this rule, so it is applied as given.

## Task 1.4 — Features and a NumPy summary
Create two new columns, then summarise four numeric columns with NumPy (no loops).

In [ ]:
# lead_days = how many days before the appointment the booking was made
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# no_show = 1 for a missed appointment, 0 for attended (this is the target for Part 3)
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print("No-shows:", df["no_show"].sum(), "out of", len(df), f"({df['no_show'].mean()*100:.1f}%)")

lead_days range: 1 to 45
No-shows: 218 out of 990 (22.0%)


In [ ]:
# Turn 4 columns into a NumPy array: rows = appointments, columns = features
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("X shape:", X.shape)

# (a) Mean of each column. axis=0 collapses the rows -> one mean per column
col_means = X.mean(axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

# (b) Boolean mask: True where lead_days > 14 (lead_days is column index 3)
mask = X[:, 3] > 14
print("\nRows with lead_days > 14:", mask.sum())        # each True counts as 1
print(f"Share: {mask.mean()*100:.1f}%")                 # mean of True/False = the share

# (c) Z-scores for distance_km (column index 1). np.std = population std (NumPy default)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("\nRows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

X shape: (990, 4)
Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]

Rows with lead_days > 14: 296
Share: 29.9%

Rows with |z| > 2 for distance_km: 53


**Result:** lead_days runs from 1 to 45 days. 218 of 990 appointments were no-shows (22.0%).
Column means: age 48.17, distance 9.50 km, fee ₹635.71, lead 11.82 days.
296 bookings (29.9%) were made more than 14 days ahead. 53 rows have |z| > 2 for distance_km.

## Task 1.5 — Enrich the clinics from a live API
Get a temperature for each clinic from the free Open-Meteo API (no key needed).
The first hourly value is the midnight-UTC reading — a stand-in, not the current weather.

In [ ]:
import requests

clinics = pd.read_csv("data/clinics.csv")
URL = "https://api.open-meteo.com/v1/forecast"

def fetch_temp(lat, lon):
    """Make ONE API call. Returns two things: (temperature, reason).
    If anything goes wrong, temperature is None and reason says why."""
    params = {"latitude": lat, "longitude": lon,
              "hourly": "temperature_2m", "forecast_days": 1}
    try:
        response = requests.get(URL, params=params, timeout=10)   # give up after 10 seconds
        if response.status_code != 200:                           # check BEFORE using the reply
            return None, f"status code {response.status_code}"
        data = response.json()                                    # reply text -> Python dict
    except requests.exceptions.RequestException as e:             # no internet, timeout, etc.
        return None, f"request failed: {e}"

    if "hourly" not in data:
        return None, "no 'hourly' data in the reply"

    return data["hourly"]["temperature_2m"][0], "status 200"     # first value = midnight UTC

In [ ]:
temps = []

for _, row in clinics.iterrows():
    temp, reason = fetch_temp(row["latitude"], row["longitude"])

    if temp is None:                     # failed -> retry this clinic once
        temp, reason = fetch_temp(row["latitude"], row["longitude"])

    if temp is None:                     # still failed -> keep None, report it, carry on
        print(f"FAILED: {row['clinic_name']} -> {reason}")
    else:
        print(f"{row['clinic_name']}: {temp} °C ({reason})")

    temps.append(temp)

clinics["temp_c"] = temps
print("\nClinics with a temperature:", clinics["temp_c"].notna().sum(), "of", len(clinics))
print("temp_c range:", clinics["temp_c"].min(), "to", clinics["temp_c"].max())

Kothrud Family Clinic: 16.7 °C (status 200)
Hadapsar Care Centre: 18.0 °C (status 200)
Sitabuldi Health Point: 20.1 °C (status 200)
Malviya Nagar Clinic: 17.4 °C (status 200)
Vaishali Nagar Clinic: 17.4 °C (status 200)
Edappally Medical Centre: 24.9 °C (status 200)
Vijay Nagar Clinic: 20.1 °C (status 200)
Gomti Nagar Care Point: 16.4 °C (status 200)

Clinics with a temperature: 8 of 8
temp_c range: 16.4 to 24.9


In [ ]:
clinics.to_csv("clinics_enriched.csv", index=False)   # original 6 columns + temp_c
print(clinics)

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    16.7  
1    18.0  
2    20.1  
3    17.4  
4    17.4  
5    24.9  
6    20.1  
7    16.4  


**Result:** 8 API calls, all status 200. clinics_enriched.csv has 8 rows (6 original columns + temp_c).
Temperatures range from 16.4 to 24.9 °C (values change with the day).

## Task 1.6 — Export the clean table and write the SQL load script
Save the final 16-column table, then turn every row into a SQL INSERT statement for Part 2.

In [ ]:
# The 16 columns the brief asks for, in this exact order
cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
        "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
        "prior_no_shows", "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id")[cols].copy()

# Dates -> 'YYYY-MM-DD' text (the format MySQL's DATE type expects)
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")

# Checks BEFORE exporting
print("appointment_id is unique:", clean["appointment_id"].is_unique)
print("Distinct appointment_id:", clean["appointment_id"].nunique())
print("Missing values in the 16 columns:", clean.isna().sum().sum())

clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean.shape)


appointment_id is unique: True
Distinct appointment_id: 990
Missing values in the 16 columns: 0
Saved appointments_clean.csv: (990, 16)


In [ ]:
lines = []

# 1) Clinics FIRST: the appointments table points to clinics through a foreign key
clinics_enriched = pd.read_csv("clinics_enriched.csv")
for _, c in clinics_enriched.iterrows():
    temp = "NULL" if pd.isna(c["temp_c"]) else c["temp_c"]   # missing temperature -> NULL (no quotes)
    lines.append(
        f"INSERT INTO clinics VALUES ('{c['clinic_id']}','{c['clinic_name']}','{c['city']}',"
        f"'{c['tier']}',{c['latitude']},{c['longitude']},{temp});"
    )

# 2) Then the appointments. Text in single quotes, numbers without quotes
for _, r in clean.iterrows():
    lines.append(
        f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
        f"'{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',{r['distance_km']},"
        f"{r['fee_inr']},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',"
        f"{r['prior_no_shows']},'{r['status']}',{r['lead_days']},{r['no_show']});"
    )

# Write one statement per line
with open("load_data.sql", "w") as f:
    f.write("\n".join(lines))

print("Clinic INSERTs:", sum(line.startswith("INSERT INTO clinics") for line in lines))
print("Appointment INSERTs:", sum(line.startswith("INSERT INTO appointments") for line in lines))
print("Total INSERT statements:", len(lines))
print("\nExample lines:")
print(lines[0])
print(lines[8])

Clinic INSERTs: 8
Appointment INSERTs: 990
Total INSERT statements: 998

Example lines:
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,16.7);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);


**Result:** appointments_clean.csv has 990 rows × 16 columns, 990 unique appointment_id values and 0 missing values.
load_data.sql has 998 INSERT statements: 8 clinics first, then 990 appointments.

Note: building SQL with f-strings is fine here because this is a fixed, controlled dataset and no value contains an apostrophe.
Real applications use parameterised queries (%s placeholders) instead, to prevent SQL injection.

## Part 2 cross-check (Task 2.2): pandas vs SQL
Reproduce the SQL clinic counts with pandas groupby to confirm both tools give the same numbers.

In [2]:
import pandas as pd
appts = pd.read_csv("appointments_clean.csv")
clin = pd.read_csv("clinics_enriched.csv")

merged = appts.merge(clin, on="clinic_id")                 # same as the SQL JOIN

# no_show is 0/1, so count = number of appointments, sum = number of no-shows
by_clinic = merged.groupby(["clinic_name", "city"])["no_show"].agg(["count", "sum"])
by_clinic.columns = ["appointments", "no_shows"]
by_clinic["no_show_pct"] = (100 * by_clinic["no_shows"] / by_clinic["appointments"]).round(1)

print(by_clinic.sort_values("no_show_pct", ascending=False))

                                  appointments  no_shows  no_show_pct
clinic_name              city                                        
Gomti Nagar Care Point   Lucknow            90        30         33.3
Hadapsar Care Centre     Pune              143        39         27.3
Malviya Nagar Clinic     Jaipur            142        38         26.8
Kothrud Family Clinic    Pune              160        36         22.5
Vijay Nagar Clinic       Indore            101        21         20.8
Edappally Medical Centre Kochi             142        23         16.2
Sitabuldi Health Point   Nagpur             82        13         15.9
Vaishali Nagar Clinic    Jaipur            130        18         13.8


**Result:** the pandas counts match the SQL output exactly for all 8 clinics (990 appointments, 218 no-shows in total).